# C9 · GFF3, sistemas de coordenadas y bedtools

**Curso:** Bioinformática y Biología Computacional · Universidad EAFIT  
**Duración sugerida:** 3 horas  
**Modalidad:** explicación breve → práctica guiada → reto → evidencia reproducible

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/UniversidadEAFIT/compubiol_course/blob/master/notebooks/09_gff3_bedtools/09_gff3_bedtools.ipynb)

## Pregunta guía

Se deben extraer promotores y exones de genes en ambas hebras, incluidos genes cercanos al borde cromosómico. **¿Dónde aparecen los errores off-by-one y cómo se detectan?**

### Objetivos

- describir las nueve columnas de GFF3 y relaciones `ID`/`Parent`;
- diferenciar coordenadas GFF3 (1-based, inclusivas) y BED (0-based, semiabiertas);
- respetar hebra y límites cromosómicos;
- construir promotores en BED6;
- extraer secuencias con `bedtools getfasta -s`;
- validar manualmente casos positivos, negativos y de borde.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys
import importlib.util

REPO_URL = "https://github.com/UniversidadEAFIT/compubiol_course.git"
COLAB_DIR = Path("/content/compubiol_course")

IN_COLAB = "COLAB_RELEASE_TAG" in os.environ
if IN_COLAB and importlib.util.find_spec("Bio") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "biopython"], check=True)

if IN_COLAB and not COLAB_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(COLAB_DIR)], check=True)
    os.chdir(COLAB_DIR)

start = Path.cwd().resolve()
ROOT = next((p for p in [start, *start.parents] if (p / "data").is_dir() and (p / "notebooks").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError(
        "No se encontró la raíz del curso. Ejecute el notebook desde el repositorio clonado."
    )
os.chdir(ROOT)
os.environ["COURSE_ROOT"] = str(ROOT)
print(f"Raíz del curso: {ROOT}")

## 1. Las nueve columnas

`seqid, source, type, start, end, score, strand, phase, attributes`. Los atributos conectan jerarquías: `gene → mRNA → exon/CDS`. No todos los proveedores usan exactamente los mismos nombres; inspeccione y valide.

In [ ]:
import pandas as pd
cols = ["seqid", "source", "type", "start", "end", "score", "strand", "phase", "attributes"]
gff = pd.read_csv(
    ROOT / "data/module09/mini_annotations.gff3",
    sep="\t", comment="#", names=cols, dtype={"seqid": str}
)
gff.head(10)

In [ ]:
gff.groupby(["type", "strand"]).size().rename("records").reset_index()

## 2. Conversión de coordenadas

Para una característica GFF3 `start..end`:

- intervalo BED de la misma característica: `[start-1, end)`;
- longitud: `end - (start-1) = end-start+1`.

Promotor de longitud `L`:

- hebra `+`: `[max(0, start-1-L), start-1)`;
- hebra `-`: `[end, min(chrom_length, end+L))`, y luego reverse complement al extraer con `-s`.

## 3. Construir promotores de forma auditable

In [ ]:
import subprocess
out = ROOT / "results/module09/promoters_200.bed"
out.parent.mkdir(parents=True, exist_ok=True)
subprocess.run([
    "python", "scripts/module09/build_promoters.py",
    "--genome", "data/module09/mini_genome.fasta",
    "--gff", "data/module09/mini_annotations.gff3",
    "--length", "200",
    "--output", str(out),
], cwd=ROOT, check=True)
print(out.read_text())

Observe `geneA`: comienza en 21, por lo que solo hay 20 bases disponibles aguas arriba. El script recorta el intervalo al límite 0 en vez de producir coordenadas negativas.

## 4. Extraer secuencia respetando hebra

In [ ]:
import shutil, subprocess
if shutil.which("bedtools"):
    result = subprocess.run([
        "bash", "scripts/module09/extract_promoters.sh",
        "data/module09/mini_genome.fasta",
        "data/module09/mini_annotations.gff3",
        "200", "results/module09/bedtools",
    ], cwd=ROOT, text=True, capture_output=True)
    print(result.stdout)
    print(result.stderr)
else:
    print("bedtools no instalado; el BED ya puede revisarse y la extracción queda lista en el script.")

## 5. Validaciones independientes

In [ ]:
# Validar que los intervalos estén dentro de sus contigs y tengan longitud positiva.
def fasta_lengths(path):
    sizes, current = {}, None
    for line in path.read_text().splitlines():
        if line.startswith(">"):
            current = line[1:].split()[0]
            sizes[current] = 0
        elif line:
            sizes[current] += len(line.strip())
    return sizes

sizes = fasta_lengths(ROOT / "data/module09/mini_genome.fasta")
beds = pd.read_csv(out, sep="\t", names=["seqid", "start", "end", "id", "score", "strand"])
beds["valid_bounds"] = beds.apply(lambda r: 0 <= r.start < r.end <= sizes[r.seqid], axis=1)
beds["length"] = beds.end - beds.start
beds

### Checkpoint

Compruebe manualmente:

1. `geneA` en `+`, recortado por el inicio del contig;
2. `geneB` en `-`, cuyo promotor está después del `end` genómico y se reverse-complementa;
3. `geneC` con dos transcritos, pero un solo registro `gene`: decida si el promotor debe definirse por gen o por transcrito.

## Reto

Extraiga exones y promotores, conserve identificadores y valide tres casos. Luego busque un motivo cis simple y explique por qué una coincidencia de secuencia no demuestra regulación funcional.

Referencia: especificación GFF3, documentación NCBI y `bedtools getfasta`.